# Notebook : Modélisation

## 1. Introduction & Contexte du Projet
Dans le cadre de l'Appel à Manifestation d'Intérêt (AMI) lancé par l'ADEME, notre équipe **CesiCDP** se mobilise pour concevoir des solutions innovantes de Mobilité Multimodale Intelligente. 

Aujourd'hui, l'optimisation de la logistique urbaine représente un enjeu écologique et économique majeur. Une flotte mal optimisée engendre des kilomètres parcourus à vide, une surconsommation de carburant, et une augmentation drastique des émissions de gaz à effet de serre.

**Notre objectif** : Concevoir un algorithme capable de planifier des tournées optimales pour un ensemble de livraisons, en minimisant la distance totale parcourue et, par extension, l'empreinte carbone. Pour que notre démonstrateur soit ancré dans la réalité du terrain, le problème algorithmique de base (le problème du voyageur de commerce) n'est pas suffisant. Nous devons l'enrichir.

## 2. Choix et Justification des Contraintes Métiers

Pour répondre aux exigences des territoires et proposer une solution réaliste, nous avons choisi d'intégrer **deux contraintes majeures** à notre modélisation :

* **Contrainte 1 : L'utilisation de plusieurs véhicules (Flotte de véhicules)**
* **Contrainte 2 : La capacité maximale des véhicules (Charge utile)**

### Pourquoi ce choix ? (Exemple concret)
Afin de pouvoir réaliser une solution la plus réaliste possible, il est nécessaire de devoir prendre en compte ces 2 contraintes car ce sont les plus courantes dans les problèmes de livraison de colis et celle qui sont les plus réalistent. Si nous ne prenons pas ces contraintes en compte, alors on risque de faire des calculs bons mais peu applicable dans une situation d'entreprise réel.

## 3. Modélisation Formelle du Problème (CVRP)

Nous allons formaliser mathématiquement notre problème à l'aide de la théorie des graphes.

### Les Données (Paramètres)
* Soit un graphe complet orienté $G = (V, E)$.
* $V = \{0, 1, 2, ..., n\}$ représente l'ensemble des sommets. Le sommet $0$ correspond au **dépôt** (point de départ et d'arrivée de tous les véhicules), et les sommets $1$ à $n$ représentent les **villes ou clients** à visiter.
* $E = \{(i, j) \mid i, j \in V, i \neq j\}$ représente l'ensemble des routes (arêtes) reliant les sommets.
* $c_{ij}$ est le **coût** (distance, temps ou consommation CO2) pour parcourir l'arête de $i$ vers $j$.
* $K$ est le **nombre de véhicules** homogènes disponibles au dépôt.
* $Q$ est la **capacité maximale** de chaque véhicule.
* $q_i$ est la **demande** (poids ou volume du colis) requise par le client $i$. Le dépôt a une demande nulle ($q_0 = 0$).

### Les Variables de Décision
Pour que l'algorithme "décide" du trajet, nous introduisons deux variables :
* Une variable binaire $x_{ijk}$ qui vaut $1$ si le véhicule $k$ emprunte la route allant de la ville $i$ à la ville $j$, et $0$ sinon.
* Une variable continue $u_i$ qui représente la charge cumulée du véhicule après avoir visité le client $i$ (utilisée pour éliminer les sous-tournées déconnectées du dépôt).

### La Fonction Objectif
Nous cherchons à **minimiser le coût total** de l'ensemble des tournées. Mathématiquement, cela se traduit par :

$$\text{Minimiser} \sum_{k=1}^{K} \sum_{i=0}^{n} \sum_{j=0}^{n} c_{ij} x_{ijk}$$

### Les Contraintes Mathématiques

**1. Chaque client doit être visité exactement une fois par un seul véhicule :**
$$\sum_{k=1}^{K} \sum_{i=0}^{n} x_{ijk} = 1 \quad \forall j \in \{1, ..., n\}$$

**2. Tout véhicule qui entre chez un client doit en repartir (Continuité du trajet) :**
$$\sum_{i=0}^{n} x_{ipk} - \sum_{j=0}^{n} x_{pjk} = 0 \quad \forall p \in \{1, ..., n\}, \forall k \in \{1, ..., K\}$$

**3. Les véhicules partent du dépôt et y reviennent :**
$$\sum_{j=1}^{n} x_{0jk} = 1 \quad \forall k \in \{1, ..., K\}$$
$$\sum_{i=1}^{n} x_{i0k} = 1 \quad \forall k \in \{1, ..., K\}$$

**4. Respect de la capacité maximale du véhicule (Contrainte de Capacité) :**
$$\sum_{i=0}^{n} \sum_{j=1}^{n} q_j x_{ijk} \leq Q \quad \forall k \in \{1, ..., K\}$$

**5. Élimination des sous-tournées (Adaptation des contraintes MTZ) :**
$$u_i - u_j + Q \sum_{k=1}^{K} x_{ijk} \leq Q - q_j \quad \forall i, j \in \{1, ..., n\}, i \neq j$$

## 4. Analyse de la Complexité Théorique

L'étude de la complexité est cruciale car elle détermine la faisabilité algorithmique de notre projet sur de grands territoires.

### Le problème de base : Le TSP
Notre problème dérive directement du Problème du Voyageur de Commerce (TSP - *Travelling Salesman Problem*). La version décisionnelle du TSP consiste à se demander : *"Existe-t-il une tournée passant par toutes les villes une seule fois dont le coût total est inférieur ou égal à une valeur L ?"*
Il a été formellement prouvé par Richard M. Karp en 1972 que le TSP appartient à la classe des problèmes **NP-complets**. Cela signifie qu'il n'existe, à ce jour, aucun algorithme capable de trouver la solution exacte en un temps polynomial (un temps raisonnable) à mesure que le nombre de villes augmente.

### Complexité du CVRP (Notre problème)
Le problème de routage de véhicules avec capacités (CVRP) que nous avons modélisé est une généralisation du TSP. 
En effet, si l'on fixe le nombre de véhicules $K = 1$ et que l'on suppose que la capacité $Q$ est infinie (ou supérieure à la somme de toutes les demandes), notre CVRP devient très exactement un TSP.
Puisque le TSP est NP-difficile au sens de l'optimisation, et qu'il constitue un cas particulier du CVRP, alors **le CVRP est intrinsèquement NP-difficile**.

**Conséquence pour le projet ADEME :**
La nature NP-difficile de ce problème justifie l'approche que nous adopterons dans les prochaines phases du projet. Pour les petites instances, une résolution mathématique exacte sera possible. Cependant, pour des instances réalistes à l'échelle d'une région, nous serons dans l'obligation de concevoir des heuristiques ou méta-heuristiques.

## 5. Références Bibliographiques

* **Dantzig, G. B., & Ramser, J. H. (1959).** *The Truck Dispatching Problem*. Management Science, 6(1), 80-91.
* **Karp, R. M. (1972).** *Reducibility among Combinatorial Problems*. Dans Complexity of Computer Computations. Springer.
* **Toth, P., & Vigo, D. (2014).** *Vehicle Routing: Problems, Methods, and Applications*. Society for Industrial and Applied Mathematics (SIAM).

# Livrable 2 
# 1.Génération des Instances Aléatoires

Afin de tester nos algorithmes de manière pertinente, nous devons générer des jeux de données (instances) qui simulent le tissu urbain d'une agglomération. 

**Justification des paramètres :**
* **Topologie :** Nous utiliserons un graphe complet où les coordonnées des clients sont tirées aléatoirement sur une grille 2D (simulant une carte avec des coordonnées GPS simplifiées). La distance sera euclidienne.
* **Tailles des instances :** Pour la phase de prototypage, nous générerons des instances de 20 à 50 clients. Le dépôt sera situé au centre de la zone.
* **Capacité et Demandes :** Pour illustrer nos vélos-cargos, nous fixerons une capacité maximale $Q = 50$ kg. La demande de chaque client sera un poids aléatoire compris entre 5 kg et 15 kg.

In [2]:
import math
import random
import matplotlib.pyplot as plt

def generate_cvrp_instance(num_clients=30, capacity=1000, grid_size=100):
    """
    Génère une instance aléatoire pour le problème CVRP.
    
    Args:
        num_clients (int): Nombre de clients à livrer.
        capacity (int): Capacité maximale d'un véhicule.
        grid_size (int): Taille de la grille (x, y) pour les coordonnées.
        
    Returns:
        dict: Contient les coordonnées, les demandes, la capacité et la matrice des distances.
    """
    # Le dépôt (index 0) est placé au centre de la zone
    coords = [(grid_size // 2, grid_size // 2)]
    # Génération des coordonnées des clients
    coords += [(random.randint(0, grid_size), random.randint(0, grid_size)) for _ in range(num_clients)]
    
    # Demande du dépôt est 0, celles des clients entre 5 et 15
    demands = [0] + [random.randint(5, 15) for _ in range(num_clients)]
    
    # Calcul de la matrice des distances (Euclidienne)
    n = len(coords)
    distance_matrix = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = coords[i][0] - coords[j][0]
                dy = coords[i][1] - coords[j][1]
                distance_matrix[i][j] = math.sqrt(dx**2 + dy**2)
                
    return {
        "coords": coords,
        "demands": demands,
        "capacity": capacity,
        "distance_matrix": distance_matrix,
        "num_clients": num_clients
    }

# --- Test de la génération ---
random.seed(42) # Pour la reproductibilité lors de l'évaluation ADEME
instance = generate_cvrp_instance()
print(f"Instance générée avec 1 dépôt et {instance['num_clients']} clients.")
print(f"Capacité des véhicules : {instance['capacity']} kg.")

Instance générée avec 1 dépôt et 30 clients.
Capacité des véhicules : 1000 kg.


---
# 2. Première Méthode : Heuristique Constructive (Plus Proche Voisin)

La première étape de notre système d'optimisation consiste à trouver une solution rapidement. Nous utilisons l'algorithme glouton du **Plus Proche Voisin (Nearest Neighbor)** adapté au CVRP.

**Le principe est simple et imite le comportement humain :**
1. Le livreur part du dépôt.
2. Il regarde quel est le client le plus proche de sa position actuelle.
3. Si la cargaison restante permet de livrer ce client, il y va. Sinon, il retourne au dépôt pour recharger son vélo-cargo, clôturant ainsi la sous-tournée.
4. L'opération se répète jusqu'à ce que tous les clients soient livrés.

In [3]:
def solve_nearest_neighbor(instance):
    """
    Résout le CVRP avec l'heuristique du Plus Proche Voisin.
    """
    dist = instance["distance_matrix"]
    demands = instance["demands"]
    capacity = instance["capacity"]
    
    unvisited = set(range(1, instance["num_clients"] + 1))
    routes = []
    total_distance = 0.0
    
    while unvisited:
        current_node = 0 # On part du dépôt
        current_load = 0
        current_route = [0]
        
        while unvisited:
            # Chercher le voisin non visité le plus proche
            nearest_neighbor = None
            min_dist = float('inf')
            
            for client in unvisited:
                # Vérifier la contrainte de capacité
                if current_load + demands[client] <= capacity:
                    if dist[current_node][client] < min_dist:
                        min_dist = dist[current_node][client]
                        nearest_neighbor = client
            
            # Si aucun client ne peut être ajouté (capacité atteinte), on rentre
            if nearest_neighbor is None:
                break
                
            # Mettre à jour l'état
            current_route.append(nearest_neighbor)
            unvisited.remove(nearest_neighbor)
            total_distance += min_dist
            current_load += demands[nearest_neighbor]
            current_node = nearest_neighbor
            
        # Retour au dépôt
        current_route.append(0)
        total_distance += dist[current_node][0]
        routes.append(current_route)
        
    return routes, total_distance

routes_nn, cost_nn = solve_nearest_neighbor(instance)
print(f"Coût total (Plus Proche Voisin) : {cost_nn:.2f} km")
print(f"Nombre de véhicules utilisés : {len(routes_nn)}")

Coût total (Plus Proche Voisin) : 562.56 km
Nombre de véhicules utilisés : 1


---
# 3. Seconde Méthode : Métaheuristique d'Amélioration (Recherche Locale 2-Opt)

L'algorithme du Plus Proche Voisin est rapide, mais il est "myope". Il fait des choix locaux qui peuvent s'avérer désastreux sur la fin du trajet (croisement de routes). 
Pour notre démonstrateur ADEME, nous devons proposer une solution plus intelligente. Nous allons appliquer un algorithme de **Recherche Locale (2-Opt)** sur chaque tournée générée.

**Principe du 2-Opt :**
L'algorithme supprime deux arêtes (routes) qui se croisent dans une tournée et les reconnecte d'une manière différente. Si la nouvelle configuration diminue la distance totale, elle est conservée. Ce processus est répété jusqu'à ce qu'aucune amélioration ne soit possible.

In [5]:
def two_opt_route(route, dist_matrix):
    """
    Applique l'algorithme 2-Opt sur une seule tournée pour l'optimiser.
    """
    best_route = route.copy()
    improved = True
    
    while improved:
        improved = False
        # On ne modifie pas le premier et le dernier élément (le dépôt)
        for i in range(1, len(best_route) - 2):
            for j in range(i + 1, len(best_route) - 1):
                if j - i == 1: continue # Sommets adjacents
                
                # Calcul de la différence de distance si on inverse le segment
                current_cost = dist_matrix[best_route[i-1]][best_route[i]] + dist_matrix[best_route[j]][best_route[j+1]]
                new_cost = dist_matrix[best_route[i-1]][best_route[j]] + dist_matrix[best_route[i]][best_route[j+1]]
                
                if new_cost < current_cost:
                    # Inversion du segment [i:j]
                    best_route[i:j+1] = reversed(best_route[i:j+1])
                    improved = True
                    break # On recommence la recherche avec la nouvelle route
            if improved:
                break
                
    return best_route

def calculate_total_cost(routes, dist_matrix):
    """Calcule le coût total d'un ensemble de routes."""
    total = 0.0
    for route in routes:
        for i in range(len(route) - 1):
            total += dist_matrix[route[i]][route[i+1]]
    return total

def solve_with_2opt(routes_initiales, instance):
    """
    Optimise toutes les routes d'une solution initiale avec 2-Opt.
    """
    dist = instance["distance_matrix"]
    optimized_routes = []
    
    for route in routes_initiales:
        opt_route = two_opt_route(route, dist)
        optimized_routes.append(opt_route)
        
    final_cost = calculate_total_cost(optimized_routes, dist)
    return optimized_routes, final_cost

routes_2opt, cost_2opt = solve_with_2opt(routes_nn, instance)
economie = cost_nn - cost_2opt

print(f"Coût initial (NN) : {cost_nn:.2f} km")
print(f"Coût optimisé (2-Opt) : {cost_2opt:.2f} km")
print(f"Économie réalisée (Carbone/Carburant) : {economie:.2f} km")

Coût initial (NN) : 562.56 km
Coût optimisé (2-Opt) : 546.93 km
Économie réalisée (Carbone/Carburant) : 15.63 km


### Conclusion de l'Implémentation
La combinaison d'une heuristique de construction (pour créer des flottes respectant les capacités) et d'une métaheuristique d'amélioration (pour démêler les routes inefficaces) nous fournit un algorithme robuste. Cette architecture modulaire sera notre socle pour le **Livrable 3 : L'étude statistique expérimentale**.